# 📘 Class Activity: Byte-Pair Encoding (BPE) Sub-Word Tokenizer
### Course: Theory of Automata  |  Topic 2: BPE Sub-Word Tokenizer
**Total Marks: 10** (Question 1: 5 marks, Question 2: 5 marks)

**Instructions for students:**
- Read the concept explanation below carefully before attempting the questions.
- Write your code only inside the sections marked `# TODO: write your code here`.
- Do **not** modify the provided helper code (BPE training code) or the test-case cells.
- Run the **Main** cell at the end to see your functions in action.
- Run the test-case cells — if all assertions pass without error, your solution is correct.


## 🧠 Concept Explanation

**What is Byte-Pair Encoding (BPE)?**

BPE is a **sub-word tokenization** algorithm originally developed for data compression and later adapted for NLP (used in GPT, RoBERTa, etc.). Instead of splitting text into full words or single characters, BPE learns a vocabulary of the *most frequently occurring character sequences*, letting common words stay whole while rare/unseen words get broken into meaningful sub-word pieces.

**Why do we need it?**
- Word-level tokenizers fail on rare or unseen words (out-of-vocabulary problem).
- Character-level tokenizers avoid OOV issues but produce very long sequences and lose word-level meaning.
- BPE is a middle ground: frequent words stay as single tokens, rare words are split into reusable sub-word units.

**How the BPE algorithm works (training phase):**
1. Start with a corpus where every word is split into individual characters, with a special end-of-word marker `</w>`.
   - e.g. `"low"` → `l o w </w>`
2. Count the frequency of every **adjacent symbol pair** in the corpus.
3. Find the **most frequent pair** and merge it into a single new symbol everywhere it occurs.
4. Repeat steps 2–3 for a fixed number of merge operations. Each merge is recorded, in order, in a **merge list**.
5. The final merge list defines the learned vocabulary/tokenizer.

**How BPE tokenizes a *new* word (encoding phase):**
- Split the new word into characters + `</w>`.
- Repeatedly apply the **learned merges**, in the order they were learned, to the adjacent symbol pairs — merging whichever pair appears earliest in the merge list — until no more merges apply.
- What remains are the word's sub-word tokens.

**Token Compression Ratio**

Once text is tokenized, we can measure how efficiently BPE compresses it:

$$\text{Compression Ratio} = \frac{\text{Total number of characters in the text (excluding spaces)}}{\text{Total number of BPE tokens produced}}$$

A **higher ratio** means BPE is packing more characters into fewer tokens (more compression/efficiency). A ratio of 1.0 would mean BPE produced one token per character (no compression at all, like raw character-level tokenization).


## 🛠️ Provided Code For Understanding.

The cell below implements the **training phase** of BPE: it learns merge rules from a small word-frequency corpus. This is provided so you can focus on the *application* of BPE (the activity questions) rather than re-deriving training from scratch.

In [ ]:
import re
import collections


def get_stats(vocab):
    """Count frequency of every adjacent symbol pair in the vocab."""
    pairs = collections.defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i + 1])] += freq
    return pairs


def merge_vocab(pair, vocab):
    """Merge all occurrences of `pair` in the vocab into a single symbol."""
    new_vocab = {}
    bigram = re.escape(' '.join(pair))
    pattern = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
    for word in vocab:
        new_word = pattern.sub(''.join(pair), word)
        new_vocab[new_word] = vocab[word]
    return new_vocab


def train_bpe(corpus, num_merges):
    """
    Train a BPE tokenizer on a word-frequency corpus.

    corpus: dict mapping word -> frequency, e.g. {"low": 5, "lower": 2}
    num_merges: number of merge operations to learn

    Returns: (merges, vocab)
      merges -> ordered list of merged symbol pairs, e.g. [('e','s'), ('es','t'), ...]
      vocab  -> final character-split vocabulary after all merges
    """
    vocab = {' '.join(list(word)) + ' </w>': freq for word, freq in corpus.items()}
    merges = []
    for _ in range(num_merges):
        pairs = get_stats(vocab)
        if not pairs:
            break
        best = max(pairs, key=pairs.get)
        vocab = merge_vocab(best, vocab)
        merges.append(best)
    return merges, vocab


# Sample training corpus (word -> frequency)
training_corpus = {"low": 5, "lower": 2, "newest": 6, "widest": 3}

learned_merges, final_vocab = train_bpe(training_corpus, num_merges=10)

print("Learned merges (in the order they were learned):")
for rank, pair in enumerate(learned_merges):
    print(f"  {rank}: {pair}")

print("\nFinal character-split vocabulary:")
for word, freq in final_vocab.items():
    print(f"  {word!r} : freq={freq}")


Learned merges (in the order they were learned):
  0: ('e', 's')
  1: ('es', 't')
  2: ('est', '</w>')
  3: ('l', 'o')
  4: ('lo', 'w')
  5: ('n', 'e')
  6: ('ne', 'w')
  7: ('new', 'est</w>')
  8: ('low', '</w>')
  9: ('w', 'i')

Final character-split vocabulary:
  'low</w>' : freq=5
  'low e r </w>' : freq=2
  'newest</w>' : freq=6
  'wi d est</w>' : freq=3


---
## ❓ Question 1 (5 Marks)

Implement the function **`bpe_tokenize_word(word, merges)`** which takes:
- `word`: a plain string (a new word, possibly not in the training corpus), and
- `merges`: the ordered list of learned merges (as produced by `train_bpe` above),

and returns a **list of sub-word tokens** for that word, obtained by applying the BPE encoding procedure:

1. Split `word` into a list of characters, and append the end-of-word marker `'</w>'` at the end.
2. Repeatedly find, among all adjacent symbol pairs currently in the list, the one pair that appears **earliest** in `merges` (i.e. has the lowest index/rank in the merges list).
3. Merge that pair into a single symbol (string concatenation of the two symbols).
4. Repeat steps 2–3 until no adjacent pair in the list is found in `merges` anymore.
5. Return the resulting list of symbols (the sub-word tokens).

**Example:** with the `learned_merges` trained above, `bpe_tokenize_word("lowest", learned_merges)` should return `['low', 'est</w>']`.

In [ ]:
def bpe_tokenize_word(word, merges):
    """
    Apply learned BPE merges to tokenize a single word into sub-words.

    Parameters
    ----------
    word : str
        The word to tokenize, e.g. "lowest"
    merges : list of tuple
        Ordered list of learned merge pairs, e.g. [('e','s'), ('es','t'), ...]

    Returns
    -------
    list of str
        The sub-word tokens for `word`, e.g. ['low', 'est</w>']
    """
    # TODO: write your code here
    symbols = list(word) + ["</w>"]

    merge_rank = {pair: i for i, pair in enumerate(merges)}

    while True:
        pairs = [(symbols[i], symbols[i + 1])
                 for i in range(len(symbols) - 1)]

        valid_pairs = [pair for pair in pairs if pair in merge_rank]

        if not valid_pairs:
            break

        best_pair = min(valid_pairs, key=lambda pair: merge_rank[pair])

        new_symbols = []
        i = 0

        while i < len(symbols):
            if (i < len(symbols) - 1 and
                    (symbols[i], symbols[i + 1]) == best_pair):
                new_symbols.append(symbols[i] + symbols[i + 1])
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1

        symbols = new_symbols

    return symbols


### ✅ Test Cases — Question 1
Run this cell after implementing `bpe_tokenize_word`. If every `assert` passes silently, the implementation is correct.

In [ ]:
# --- TA / Auto-grading test cases for Question 1 ---
_test_merges, _ = train_bpe({"low": 5, "lower": 2, "newest": 6, "widest": 3}, num_merges=10)

test_cases_q1 = [
    ("lowest", ['low', 'est</w>']),
    ("newer",  ['new', 'e', 'r', '</w>']),
    ("wide",   ['wi', 'd', 'e', '</w>']),
    ("low",    ['low</w>']),
]

passed = 0
for word, expected in test_cases_q1:
    result = bpe_tokenize_word(word, _test_merges)
    status = "PASS" if result == expected else "FAIL"
    if status == "PASS":
        passed += 1
    print(f"[{status}] bpe_tokenize_word({word!r}) -> {result}  (expected {expected})")

print(f"\n{passed}/{len(test_cases_q1)} test cases passed for Question 1.")
assert passed == len(test_cases_q1), "Question 1: some test cases failed."


[PASS] bpe_tokenize_word('lowest') -> ['low', 'est</w>']  (expected ['low', 'est</w>'])
[PASS] bpe_tokenize_word('newer') -> ['new', 'e', 'r', '</w>']  (expected ['new', 'e', 'r', '</w>'])
[PASS] bpe_tokenize_word('wide') -> ['wi', 'd', 'e', '</w>']  (expected ['wi', 'd', 'e', '</w>'])
[PASS] bpe_tokenize_word('low') -> ['low</w>']  (expected ['low</w>'])

4/4 test cases passed for Question 1.


---
## ❓ Question 2 (5 Marks)

Implement the function **`compute_compression_ratio(text, merges)`** which takes:
- `text`: a sample paragraph (a plain string, possibly multiple words separated by spaces), and
- `merges`: the ordered list of learned merges,

and returns a tuple **`(ratio, total_chars, total_tokens)`** where:
- `total_chars` = total number of characters in `text`, **excluding spaces**, summed over all words.
- `total_tokens` = total number of BPE sub-word tokens produced when every word in `text` is tokenized using `bpe_tokenize_word` (from Question 1) and the token counts are summed across all words.
- `ratio` = `total_chars / total_tokens` (the **token compression ratio**), rounded to 4 decimal places.

**Hint:** split `text` on whitespace to get the words, call your `bpe_tokenize_word` function from Question 1 on each word, and accumulate the counts.

In [ ]:
def compute_compression_ratio(text, merges):
    """
    Compute the BPE token compression ratio for a sample paragraph.

    Parameters
    ----------
    text : str
        A sample paragraph, e.g. "the lowest newest widest lower low newer"
    merges : list of tuple
        Ordered list of learned merge pairs.

    Returns
    -------
    tuple(float, int, int)
        (compression_ratio rounded to 4 dp, total_chars, total_tokens)
    """
    # TODO: write your code here
    words = text.split()

    total_chars = 0
    total_tokens = 0

    for word in words:
        total_chars += len(word)
        tokens = bpe_tokenize_word(word, merges)
        total_tokens += len(tokens)

    ratio = total_chars / total_tokens

    return (round(ratio, 4), total_chars, total_tokens)


### ✅ Test Cases — Question 2
Run this cell after implementing `compute_compression_ratio`. If every `assert` passes silently, the implementation is correct.

In [ ]:
# --- TA / Auto-grading test cases for Question 2 ---
_test_merges, _ = train_bpe({"low": 5, "lower": 2, "newest": 6, "widest": 3}, num_merges=10)

sample_paragraph = "the lowest newest widest lower low newer"
expected_chars = 34
expected_tokens = 19
expected_ratio = 1.7895

ratio, total_chars, total_tokens = compute_compression_ratio(sample_paragraph, _test_merges)

print(f"Paragraph: {sample_paragraph!r}")
print(f"total_chars  -> got {total_chars}, expected {expected_chars}")
print(f"total_tokens -> got {total_tokens}, expected {expected_tokens}")
print(f"ratio        -> got {ratio}, expected {expected_ratio}")

assert total_chars == expected_chars, "total_chars is incorrect."
assert total_tokens == expected_tokens, "total_tokens is incorrect."
assert abs(ratio - expected_ratio) < 1e-3, "compression ratio is incorrect."
print("\nAll Question 2 test cases passed!")


Paragraph: 'the lowest newest widest lower low newer'
total_chars  -> got 34, expected 34
total_tokens -> got 19, expected 19
ratio        -> got 1.7895, expected 1.7895

All Question 2 test cases passed!


---
## 🚀 Main — Demonstration

This cell trains the tokenizer, then uses **both** of your functions on a fresh example so you can see the full pipeline end-to-end.

In [ ]:
# ===== MAIN DEMO =====
corpus = {"low": 5, "lower": 2, "newest": 6, "widest": 3}
merges, vocab = train_bpe(corpus, num_merges=10)

print("### Step 1: Trained BPE merges ###")
for rank, pair in enumerate(merges):
    print(f"  {rank}: {pair}")

print("\n### Step 2: Tokenizing individual words (Question 1) ###")
demo_words = ["lowest", "newer", "wide", "low"]
for w in demo_words:
    tokens = bpe_tokenize_word(w, merges)
    print(f"  {w!r:10s} -> {tokens}")

print("\n### Step 3: Compression ratio on a paragraph (Question 2) ###")
paragraph = "the lowest newest widest lower low newer"
ratio, total_chars, total_tokens = compute_compression_ratio(paragraph, merges)
print(f"  Paragraph      : {paragraph!r}")
print(f"  Total chars    : {total_chars}")
print(f"  Total tokens   : {total_tokens}")
print(f"  Compression ratio (chars/token): {ratio}")


### Step 1: Trained BPE merges ###
  0: ('e', 's')
  1: ('es', 't')
  2: ('est', '</w>')
  3: ('l', 'o')
  4: ('lo', 'w')
  5: ('n', 'e')
  6: ('ne', 'w')
  7: ('new', 'est</w>')
  8: ('low', '</w>')
  9: ('w', 'i')

### Step 2: Tokenizing individual words (Question 1) ###
  'lowest'   -> ['low', 'est</w>']
  'newer'    -> ['new', 'e', 'r', '</w>']
  'wide'     -> ['wi', 'd', 'e', '</w>']
  'low'      -> ['low</w>']

### Step 3: Compression ratio on a paragraph (Question 2) ###
  Paragraph      : 'the lowest newest widest lower low newer'
  Total chars    : 34
  Total tokens   : 19
  Compression ratio (chars/token): 1.7895


### Conceptual Understanding

BPE merges frequently occurring characters and sub-word sequences to create larger tokens, making text representation more efficient.
It can represent common words with fewer tokens while still handling rare words by breaking them into smaller sub-word units.
The compression ratio indicates the average number of characters represented by each BPE token.